# Generate Preliminary Harmonization Mappings

**Step 2 of 3** in the BioData Catalyst harmonization workflow:

1. [Download Studies Metadata](./download_studies_data_dictionaries.ipynb) — fetch preharmonized
   PFBs from BDC and extract the dbGaP data dictionary and variable report XML files.
2. **Generate Preliminary Harmonization Mappings** (this notebook) — map each study variable to
   ranked candidate slots in the
   [BioData Catalyst Harmonized Model (bdchm)](https://github.com/RTIInternational/NHLBI-BDC-DMC-HM).
3. [Review Harmonization Suggestions](./review_harmonization_suggestions.ipynb) — accept or
   skip candidates one variable at a time.

**How it works:** `MultiPromptSimilaritySearch` embeds the bdchm schema four times, once per
prompt variant (A/B/C/D), into separate ChromaDB collections. Each source variable is queried
against all four using that variant's wording, results are merged by target slot keeping the
best similarity, and the top `top_n_suggestions` candidates per variable are written to a CSV.

No single wording wins everywhere, which is why the variants are combined: enum values help
match categorical variables, while measurement units help match lab results.

> **Before you start:** the first run downloads a ~1.3 GB embedding model and embeds the schema
> four times. Expect several minutes of setup on a GPU, considerably longer on CPU, then roughly
> a few variables per second while mapping. Mapping rows are written as they are produced, so
> you can watch the output CSV grow.

# Setup

## Libraries

Install with `pip install ai-harmonization chromadb sentence_transformers torch`.

In [ ]:
# Uncomment to install libraries (requires Python 3.12+)
#!pip -q install ai-harmonization chromadb sentence_transformers torch

In [ ]:
import csv
import glob
import os
import time

import pandas as pd
import requests
import torch
from IPython.display import display

from ai_harmonization.harmonization_approaches.embeddings import BGEEmbeddings

from ai_harmonization.dbgap import (
    CSV_HEADERS,
    find_study_metadata_files,
    generate_variable_mappings,
    summarize_rank1_similarity,
)
from ai_harmonization.formatters import (
    get_node_property_as_name_description,
    get_node_property_as_name_type_description,
    get_node_property_as_name_type_description_values,
    get_node_property_as_name_type_values,
)
from ai_harmonization.simple_data_model import SimpleDataModel
from ai_harmonization.styles import style_mapping_quality_summary
from ai_harmonization.harmonization_approaches.similarity_inmem import (
    MultiPromptSimilaritySearch,
)

## Configuration

Set the studies to map, how many candidates to keep per variable, the embedding model, and the
target schema version.

`selected_studies` must name studies already downloaded by step 1 — that is, directories under
`inputs/studies/`. It defaults to the same example study step 1 downloads.

`target_schema_version` drives everything downstream: the download URL, the cache filename, and
the ChromaDB collection names. Bumping it therefore re-downloads the schema and rebuilds the
vector index automatically, instead of silently reusing collections built from the old schema.

In [ ]:
# Studies to map. Must already be downloaded by step 1.
selected_studies = [
    'phs000704.v1.p1.c1',
]

# How many candidate target slots to keep per source variable. This also sets how
# many accept buttons the review notebook renders.
top_n_suggestions = 10

# Bio-domain fine-tuned embedding model, downloaded from Hugging Face on first use.
model_name = 'uc-ctds/bge-large-en-v1.5-bio-mapping'

# bdchm release to harmonize against. Pinned to a tag for reproducibility; use
# 'refs/heads/main' below instead of a tag if you need the unreleased schema.
target_schema_version = 'v1.4.0'
target_schema_url = (
    'https://raw.githubusercontent.com/RTIInternational/NHLBI-BDC-DMC-HM/'
    f'refs/tags/{target_schema_version}/src/bdchm/schema/bdchm.yaml'
)

# Rebuild the vector collections even when they already exist. Changing
# target_schema_version already forces a rebuild via the collection name, so this
# is only needed after editing a prompt variant formatter.
force_recreation = False

## Inputs / Outputs

Reads the dbGaP XMLs that step 1 wrote to `inputs/studies/{study_id}/metadata/`, and writes one
`{study_id}_preliminary_mappings.csv` per study into `outputs/`.

The schema is cached per version at `inputs/target_schema_{version}.yaml`, so several bdchm
releases can sit side by side without clobbering each other.

In [ ]:
inputs_dir = './inputs'
outputs_dir = './outputs'
studies_dir = os.path.join(inputs_dir, 'studies')
target_schema_path = os.path.join(inputs_dir, f'target_schema_{target_schema_version}.yaml')

# One collection per prompt variant per schema version, so bumping the version
# cannot silently reuse an index built from the previous schema.
collection_prefix = f'bdchm_{target_schema_version.replace(".", "_")}'

os.makedirs(outputs_dir, exist_ok=True)

missing = [
    study_id for study_id in selected_studies
    if not os.path.isdir(os.path.join(studies_dir, study_id, 'metadata'))
]
if missing:
    print('No downloaded metadata for: ' + ', '.join(missing))
    print('Run download_studies_data_dictionaries.ipynb first.')
else:
    print(f'{len(selected_studies)} study(ies) ready to map: {", ".join(selected_studies)}')
print(f'Target schema: bdchm {target_schema_version} -> {target_schema_path}')
print(f'Collections:   {collection_prefix}_variant_a … _variant_d')

# Get BioData Catalyst Harmonized Model (bdchm)

Download the [bdchm](https://github.com/RTIInternational/NHLBI-BDC-DMC-HM) LinkML YAML schema
for the pinned release and parse it into a `SimpleDataModel`, resolving class inheritance and
enum values.

The file is cached per version, so this downloads once per release. Bumping
`target_schema_version` fetches the new one and builds fresh collections beside the old.

In [ ]:
# Download the schema file if it doesn't already exist on disk
if os.path.exists(target_schema_path):
    print(f'Schema file found locally at {target_schema_path!r}. Skipping download.')
else:
    print(f'Schema file not found locally at {target_schema_path}. Downloading from {target_schema_url}')
    response = requests.get(target_schema_url, timeout=15)
    response.raise_for_status()
    with open(target_schema_path, 'w', encoding='utf-8') as f:
        f.write(response.text)
    print(f'Saved schema to: {target_schema_path}')

In [ ]:
with open(target_schema_path, 'r', encoding='utf-8') as f:
    target_data_model = SimpleDataModel.from_linkml_yaml(f.read())

total_slots = sum(len(node.properties) for node in target_data_model.nodes)
print(f'Parsed schema: {len(target_data_model.nodes)} classes, {total_slots} slots (with inheritance)')

# Map Study Variables

Embed the target schema once per prompt variant, then query each source variable against
every variant and merge the results by target slot.

## Device Detection

The embedding model runs on GPU if available, CPU otherwise.

In [ ]:
# BGEEmbeddings auto-detects CUDA — print GPU info for reference
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('No GPU detected — running on CPU')

## Initialize Embedding Model

Load the bio-domain fine-tuned BGE model used to embed both target slots and source variables.

In [ ]:
print('Initializing embedding model...')
biomapping_ef = BGEEmbeddings(model_name=model_name)
print('Embedding model ready.')

## Build the Multi-Prompt Index

Each prompt variant embeds the target schema with a different formatter. The same formatter
is used to query source variables, keeping each comparison symmetric.

In [ ]:
harmonization_approach = MultiPromptSimilaritySearch(
    input_target_model=target_data_model,
    prompt_variants={
        'A': get_node_property_as_name_description,
        'B': get_node_property_as_name_type_description,
        'C': get_node_property_as_name_type_description_values,
        'D': get_node_property_as_name_type_values,
    },
    embedding_function=biomapping_ef,
    collection_name_prefix=collection_prefix,
    force_vectorstore_recreation=force_recreation,
)

for label, index in harmonization_approach.indexes.items():
    print(f'Prompt variant {label}: {len(index.vectorstore.get()["ids"])} entries')

In [ ]:
cell_start_time = time.time()

slot_values_lookup = {
    f'{node.name}.{prop.name}': ', '.join(prop.values) if prop.values else ''
    for node in target_data_model.nodes
    for prop in node.properties
}

for selected_study in selected_studies:
    try:
        study_files_path, dict_files, var_report_by_pht = find_study_metadata_files(
            studies_dir, selected_study
        )
    except FileNotFoundError as e:
        print(f'[! Error] {e}')
        continue

    output_csv_path = os.path.join(outputs_dir, f'{selected_study}_preliminary_mappings.csv')
    print(f'\nProcessing study: {selected_study}')
    print(f' -> Found {len(dict_files)} data dictionaries')
    print(f' -> Output: {output_csv_path}')

    total_vars = 0
    total_rows = 0

    # Rows are flushed per variable so a long run can be inspected, or resumed
    # from, without waiting for the whole study to finish.
    with open(output_csv_path, 'w', newline='', encoding='utf-8') as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=CSV_HEADERS)
        writer.writeheader()

        for rows in generate_variable_mappings(
            study_files_path, dict_files, var_report_by_pht,
            harmonization_approach, slot_values_lookup, selected_study,
            k=top_n_suggestions,
        ):
            writer.writerows(rows)
            csv_file.flush()
            total_rows += len(rows)
            total_vars += 1

    print(f' -> Saved {total_vars} variables, {total_rows} rows to {output_csv_path}')

total_elapsed = time.time() - cell_start_time
hours, rem = divmod(total_elapsed, 3600)
minutes, seconds = divmod(rem, 60)
print(f'\nTotal execution time: {int(hours)}h {int(minutes)}m {seconds:.2f}s')

# Mapping Quality Summary

Loads all `*_preliminary_mappings.csv` files from `outputs/` and computes per-study statistics based on the **rank-1** (best) suggestion for each source variable. Use this to identify studies with the strongest AI-suggested mappings for curation.

In [ ]:
csv_files = sorted(glob.glob(f'{outputs_dir}/*_preliminary_mappings.csv'))
if not csv_files:
    print('No preliminary mapping files found in', outputs_dir)
else:
    rank1 = pd.concat(
        [pd.read_csv(f, dtype={'Original Values': str, 'Target Values': str}).query('rank == 1')
         for f in csv_files],
        ignore_index=True,
    )

    summary = (rank1.groupby('study_id').apply(summarize_rank1_similarity)
               .reset_index()
               .sort_values('Mean sim (rank 1)', ascending=False)
               .reset_index(drop=True))
    summary.index += 1

    display(style_mapping_quality_summary(summary))

    print('\nTop 3 bdchm target slots per study (mean rank-1 similarity):')
    top_slots = rank1.groupby(['study_id', 'Suggested Target Node.Property'])['Similarity'].mean()
    for study_id in summary['study_id']:
        targets = ', '.join(
            f'{slot} ({sim:.3f})'
            for slot, sim in top_slots[study_id].nlargest(3).items()
        )
        print(f'  {study_id}: {targets}')

# Next Step

Open [review_harmonization_suggestions.ipynb](./review_harmonization_suggestions.ipynb) to step
through these candidates and accept or skip one variable at a time. Studies with the highest
mean rank-1 similarity in the table above are the quickest to review.